In [20]:
import pyspark
import xgboost as xgb
import pandas as pd

In [2]:
spark = pyspark.sql.SparkSession.builder.master("local[4]").appName("spark_naive").config(
            "spark.executor.memory", "15g").config("spark.driver.memory", "15g").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/01/26 16:13:49 WARN Utils: Your hostname, starfuryz2040-VirtualBox, resolves to a loopback address: 127.0.1.1; using 10.0.2.15 instead (on interface enp0s3)
26/01/26 16:13:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/01/26 16:13:52 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
spark.sql("select count(*), prelabeled_fraud from {df} group by prelabeled_fraud", df=spark.read.parquet("./silver/labeled_training_set_no_tensor.parquet")).show()

+--------+----------------+
|count(1)|prelabeled_fraud|
+--------+----------------+
|     799|               0|
|     199|               1|
+--------+----------------+



In [9]:
xgboostmodel = xgb.XGBClassifier(enable_categorical=True)

xgboostmodel.load_model('./xgboost_model.model')

In [22]:
print(xgboostmodel)

XGBClassifier(base_score='1.9548872E-1', booster='gbtree', callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None,
              feature_types=['c', 'c', 'c', 'c', 'c', 'c', 'c', 'int', 'int'],
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)


In [14]:
booster = xgboostmodel.get_booster()

In [15]:
tree_df = booster.trees_to_dataframe()

In [16]:
tree_df

,Tree,Node,ID,Feature,Split,Yes,No,Missing,Gain,Cover,Category
0,0,0,0-0,hair_color,NaN,0-2,0-1,0-1,254.802414,125.503769,"[0, 2, 3]"
1,0,1,0-1,ethnicity,NaN,0-4,0-3,0-3,81.719833,16.670927,[3]
2,0,2,0-2,weight_lbs,149.0,0-5,0-6,0-6,4.743969,108.832840,None
3,0,3,0-3,Leaf,NaN,NaN,NaN,NaN,0.473434,12.424559,NaN
4,0,4,0-4,class,NaN,0-8,0-7,0-7,1.582464,4.246368,"[0, 2, 3]"
...,...,...,...,...,...,...,...,...,...,...,...
1159,99,8,99-8,Leaf,NaN,NaN,NaN,NaN,-0.015506,1.798205,NaN
1160,99,9,99-9,Leaf,NaN,NaN,NaN,NaN,0.001401,7.389957,NaN
1161,99,10,99-10,Leaf,NaN,NaN,NaN,NaN,-0.073433,2.286202,NaN
1162,99,11,99-11,Leaf,NaN,NaN,NaN,NaN,0.060990,3.420373,NaN


In [17]:
print(tree_df['Tree'].nunique())

100


In [18]:
tree_dfs = booster.get_dump(with_stats=True, dump_format='json') # Use json or text format

# We need a more structured way to get the depth.
# The 'xgb.model.dt.tree()' function in R is simpler, but in Python we can use
# the booster.trees_to_dataframe() method if available (requires a recent version of xgboost)
try:
    # This method is very convenient for getting node information
    all_trees_df = booster.trees_to_dataframe()

    # 4. Calculate actual depth for each tree
    # The 'Depth' column is available directly in the DataFrame
    # Group by the 'Tree' ID and find the maximum depth for each tree
    tree_depths = all_trees_df.groupby('Tree')['Depth'].max()

    print("Actual depth of each tree:")
    print(tree_depths)

except AttributeError:
    # Fallback for older versions or different environments:
    print("trees_to_dataframe() method not available. Cannot automatically determine actual depths.")
    print("You can still visualize individual trees using plot_tree to inspect depths manually.")

KeyError: 'Column not found: Depth'

In [24]:
def calculate_depths_any_format(df):
    max_depths = {}

    # Identify column names based on what exists in your dataframe
    yes_col = 'Yes' if 'Yes' in df.columns else 'TrueNode'
    no_col = 'No' if 'No' in df.columns else 'FalseNode'
    node_col = 'ID' if 'ID' in df.columns else 'Node'

    for tree_id in df['Tree'].unique():
        tree_df = df[df['Tree'] == tree_id]

        # Use a dictionary to track the depth of each node in the current tree
        # The first node in each tree's subset is the root (depth 0)
        root_id = tree_df.iloc[0][node_col]
        node_depths = {root_id: 0}
        max_depth_tree = 0

        # Iterate through nodes to propagate depth
        for _, row in tree_df.iterrows():
            curr_id = row[node_col]

            # Only process split nodes (they have children); leaves have NaN in 'Feature'
            if pd.notna(row['Feature']) and row['Feature'] != 'Leaf':
                current_depth = node_depths[curr_id]

                # Get child IDs (these might be strings like '0-1', '0-2')
                true_child = row[yes_col]
                false_child = row[no_col]

                # Assign child depth = current depth + 1
                node_depths[true_child] = current_depth + 1
                node_depths[false_child] = current_depth + 1

                # Update the maximum depth tracked for this tree
                if current_depth + 1 > max_depth_tree:
                    max_depth_tree = current_depth + 1

        max_depths[tree_id] = max_depth_tree
    return max_depths

# 3. Run and print results
tree_depth_dict = calculate_depths_any_format(all_trees_df)

print("Actual depth of each tree:")
for tree_id, depth in tree_depth_dict.items():
    print(f"Tree {tree_id}: Depth {depth}")

Actual depth of each tree:
Tree 0: Depth 3
Tree 1: Depth 3
Tree 2: Depth 3
Tree 3: Depth 3
Tree 4: Depth 3
Tree 5: Depth 3
Tree 6: Depth 3
Tree 7: Depth 3
Tree 8: Depth 3
Tree 9: Depth 3
Tree 10: Depth 3
Tree 11: Depth 3
Tree 12: Depth 3
Tree 13: Depth 3
Tree 14: Depth 3
Tree 15: Depth 3
Tree 16: Depth 3
Tree 17: Depth 3
Tree 18: Depth 3
Tree 19: Depth 3
Tree 20: Depth 3
Tree 21: Depth 3
Tree 22: Depth 3
Tree 23: Depth 3
Tree 24: Depth 3
Tree 25: Depth 3
Tree 26: Depth 3
Tree 27: Depth 3
Tree 28: Depth 3
Tree 29: Depth 3
Tree 30: Depth 3
Tree 31: Depth 3
Tree 32: Depth 3
Tree 33: Depth 3
Tree 34: Depth 3
Tree 35: Depth 3
Tree 36: Depth 3
Tree 37: Depth 3
Tree 38: Depth 3
Tree 39: Depth 3
Tree 40: Depth 3
Tree 41: Depth 3
Tree 42: Depth 3
Tree 43: Depth 3
Tree 44: Depth 3
Tree 45: Depth 3
Tree 46: Depth 3
Tree 47: Depth 3
Tree 48: Depth 3
Tree 49: Depth 3
Tree 50: Depth 3
Tree 51: Depth 3
Tree 52: Depth 3
Tree 53: Depth 3
Tree 54: Depth 3
Tree 55: Depth 3
Tree 56: Depth 3
Tree 57: Depth